#  Text Feature Extraction

In [1]:
import os
import numpy as np
import pandas as pd

In [ ]:
dataset_name = 'MicroLens-100k'
os.chdir('./datasets/%s/5-core'%dataset_name)
os.getcwd()

## Load meta data

In [ ]:
mapping_csv = f'./i_id_mapping.csv'
title_csv = f'../raw/MicroLens-100k_title_en.csv'
tag_csv = f'../raw/tags_to_summary.csv'

mapping_df = pd.read_csv(mapping_csv, dtype=str, sep='\t' if '\t' in open(mapping_csv).readline() else ',')
mapping_df['asin'] = mapping_df['asin'].astype(str)
mapping_df['itemID'] = mapping_df['itemID'].astype(str)

title_df = pd.read_csv(title_csv, header=None, names=['asin', 'title'], dtype=str)
tag_df = pd.read_csv(tag_csv, header=None, names=['asin', 'tag'], dtype=str)

df = pd.merge(title_df, tag_df, on='asin', how='outer')
print("pre mapping:", df.shape)
df = pd.merge(mapping_df, df, on='asin', how='inner') 
print("after mapping:", df.shape)

df['itemID'] = df['itemID'].astype(int)
print('data loaded!')

df[:5]

pre mapping: (19738, 3)
after mapping: (17228, 4)
data loaded!


,asin,itemID,title,tag
0,9580,0,the best place to destroy the most wonderful ...,Game
1,14631,1,# King Glory national Service Dharma being sp...,Game
2,7315,2,# King glory country serve Damo old table pla...,Game
3,1955,3,MC sees creatures that replicate? Raise an ar...,Game
4,9943,4,# King of Glory served Dharma hit the peak of...,Game


In [4]:
# sentences: title + tag
title_na_df = df[df['title'].isnull()]
print(title_na_df.shape)

desc_na_df = df[df['tag'].isnull()]
print(desc_na_df.shape)

(0, 4)
(0, 4)


In [5]:

df['title'] = df['title'].fillna(" ")
df['tag'] = df['tag'].fillna(" ")
print(f'columns: {df.columns}')
print(df.head())

columns: Index(['asin', 'itemID', 'title', 'tag'], dtype='object')
    asin  itemID                                              title   tag
0   9580       0   the best place to destroy the most wonderful ...  Game
1  14631       1   # King Glory national Service Dharma being sp...  Game
2   7315       2   # King glory country serve Damo old table pla...  Game
3   1955       3   MC sees creatures that replicate? Raise an ar...  Game
4   9943       4   # King of Glory served Dharma hit the peak of...  Game


In [ ]:
sentences = []
item_ids = []
for i, row in df.iterrows():
    if row['title'] == ' ' and row['tag'] == ' ':
        continue
    else:
        sen = row['title'] + ' ' + row['tag']
        sen = sen.replace('\n', ' ')

        sentences.append(sen)
        item_ids.append(row['itemID'])

sentences[:10]

[' the best place to destroy the most wonderful team with the fastest speed Game',
 ' # King Glory national Service Dharma being sprayed vegetable is really like this Game',
 ' # King glory country serve Damo old table play Lv Bu Game',
 ' MC sees creatures that replicate? Raise an army of iron puppets for the village! Real Village Heroes # QuickHands My World # Myworld #MC  Game',
 " # King of Glory served Dharma hit the peak of ridicule. I'm back Game",
 ' this is a 2100 division robbery bureau Game',
 " campus weird talk: roommate Ajie's prank! # thrillsuspenseful # schoolyard Weirtalk Daily Sharing",
 '  Daily Sharing',
 ' made a big box of star cedars ❄️ handmade soap ~ The New Year also want to be like a small pine 🌲 hard and lucky life ✨ #Vlog daily # handmade soap cold soap Daily Sharing',
 ' too hard # assist Game']

In [7]:
course_list = df['itemID'].tolist()

assert course_list[-1] == len(course_list) - 1

In [8]:
import math
from tqdm import tqdm

def extract_raw_text_feature(text_corpus, tokenizer, model, batch_size=512):
    batch_num = math.ceil(len(text_corpus)/batch_size)
    features = []
    for batch_idx in tqdm(range(batch_num)):
        processing_text = sentences[batch_idx*batch_size: (batch_idx+1)*batch_size]
        inputs = tokenizer(processing_text, return_tensors="pt", padding=True, truncation=True, max_length=512)  # Batch size 1
        inputs = inputs.to(model.device)
        attention_mask = inputs['attention_mask']
        input_ids = inputs.input_ids
        encoded_sentence = model(input_ids, attention_mask)
        last_hidden_states = encoded_sentence.last_hidden_state
        sentence_embedding = last_hidden_states.mean(dim=1)

        if len(features) == 0:
            features = sentence_embedding.detach().cpu().numpy()
        else:
            features = np.concatenate((features, sentence_embedding.detach().cpu().numpy()), axis=0)
    
    return features

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

gpu_id = 0
device = torch.device('cuda:' + str(gpu_id) if torch.cuda.is_available() else 'cpu')


tokenizer = AutoTokenizer.from_pretrained('./pretrain_models/xlm-roberta-base')
model = AutoModel.from_pretrained('./pretrain_models/xlm-roberta-base')
model = model.to(device)

batch_size = 4
features = extract_raw_text_feature(sentences, tokenizer, model, batch_size)

print(features.shape)
features[:5]

In [ ]:
average_feature = np.mean(features, axis=0)

# check null
features_array = np.empty((df.shape[0], features.shape[-1]))
idx = 0
for item_id in df['itemID'].values:
    if item_id not in item_ids:
        features_array[item_id] = average_feature  
    else:
        features_array[item_id] = features[idx]
        idx += 1

assert features_array.shape[0] == df.shape[0]

print(features_array.shape)
features_array[:5]

(17228, 768)


array([[ 0.03670856,  0.04487617,  0.02172744, ..., -0.05372855,
         0.02952005, -0.00612068],
       [-0.00891424,  0.06211974,  0.00745388, ..., -0.04929145,
         0.0089693 ,  0.0034734 ],
       [ 0.0517681 ,  0.04307691, -0.00034944, ..., -0.08222508,
        -0.00425497, -0.0087608 ],
       [-0.01768793, -0.00703707,  0.02532643, ...,  0.01202264,
         0.00784575,  0.02249978],
       [ 0.00699221,  0.07121632, -0.00865045, ..., -0.03044506,
         0.00011655,  0.00028755]])

In [11]:
#save
np.save(os.path.join('./', 'text_feat.npy'), features_array)
print('done!')

done!


In [12]:
# load
load_txt_feat = np.load('text_feat.npy', allow_pickle=True)
print(load_txt_feat.shape)
load_txt_feat[:10]

(17228, 768)


array([[ 0.03670856,  0.04487617,  0.02172744, ..., -0.05372855,
         0.02952005, -0.00612068],
       [-0.00891424,  0.06211974,  0.00745388, ..., -0.04929145,
         0.0089693 ,  0.0034734 ],
       [ 0.0517681 ,  0.04307691, -0.00034944, ..., -0.08222508,
        -0.00425497, -0.0087608 ],
       ...,
       [ 0.06957105,  0.07756102,  0.05045132, ..., -0.0705563 ,
        -0.03044799, -0.10292941],
       [-0.01685518,  0.05251319, -0.00631875, ...,  0.09892607,
         0.02043448,  0.01454565],
       [ 0.07224486,  0.06726371,  0.00130518, ..., -0.13554922,
         0.00581509,  0.00593463]])